# Final pipeline: 30-day hospital readmission prediction

This is the submission-ready, top-to-bottom pipeline for the IBA Karachi Introduction to Machine Learning Kaggle Challenge. It builds encounter-level features from the raw competition tables, trains the three component models, produces the v19-style rank blend, and validates the required Kaggle submission schema.

**Final selected Kaggle result:** private ROC-AUC **0.75739** for the 60% base blend / 10% shallow boosted-stump / 30% enriched GAM-style model rank blend. The originally submitted file remains in `final_submissions_and_artifacts/` for auditability.

## 1. Setup and assumptions

Run from the repository root. The raw competition tables are stored in `working/`. Intermediate files are created in `working/` and ignored by Git; this notebook writes its reproduced submission under `reproducibility_runs/` so it never overwrites the canonical submitted CSV.

The historical shallow-stump component uses a compact legacy engineered feature table. This notebook rebuilds it locally from the provided raw CSVs, so no derived encounter-level table needs to be stored in Git.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
WORKING = ROOT / 'working'
ARTIFACTS = ROOT / 'final_submissions_and_artifacts'
RUN_DIR = ROOT / 'reproducibility_runs'
NOTEBOOK_OUTPUTS = RUN_DIR / 'executed_notebooks'
RUN_DIR.mkdir(exist_ok=True)
NOTEBOOK_OUTPUTS.mkdir(parents=True, exist_ok=True)

assert WORKING.exists(), f'Missing raw-data directory: {WORKING}'
assert ARTIFACTS.exists(), f'Missing artifact directory: {ARTIFACTS}'
print('Repository root:', ROOT)
print('Raw tables:', WORKING)
print('Output directory:', RUN_DIR)

In [ ]:
required_raw = [
    'train.csv', 'test.csv', 'sample_submission.csv', 'index_encounters.csv',
    'patients.csv', 'hospitals.csv', 'diagnoses.csv', 'icd10_lookup.csv',
    'lab_results.csv', 'vitals.csv', 'medications.csv', 'procedures.csv',
    'prior_utilization.csv', 'unit_transfers.csv', 'discharge_planning.csv',
]
missing = [name for name in required_raw if not (WORKING / name).exists()]
assert not missing, f'Missing raw files: {missing}'
print(f'Validated {len(required_raw)} raw competition CSV files.')

## 2. Build the one-row-per-encounter feature tables

Notebook 02 cleans and aggregates the linked clinical tables to `features2.parquet`. Notebook 03 derives temporal/severity features and writes `features3.parquet`. Both notebooks use only the provided raw CSVs.

In [ ]:
run_env = os.environ.copy()
run_env.update({'KAGGLE_DATA': str(WORKING), 'KAGGLE_WORK': str(WORKING)})

for notebook_name in ['02_feature_engineering.ipynb', '03_extra_features.ipynb']:
    source_notebook = WORKING / notebook_name
    command = [
        sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
        '--ExecutePreprocessor.timeout=-1', '--output-dir', str(NOTEBOOK_OUTPUTS),
        str(source_notebook),
    ]
    print('Running', notebook_name)
    subprocess.check_call(command, cwd=ROOT, env=run_env)

assert (WORKING / 'features2.parquet').exists()
assert (WORKING / 'features3.parquet').exists()
print('Feature tables created successfully.')

## 3. Rebuild legacy features and train component models

The shallow-stump model uses the original compact feature set, rebuilt locally in this section. The base blend combines three regularized LightGBM configurations with logistic regression; the enriched GAM-style model uses the extended feature set. All use five-fold stratified cross-validation; the training scripts save their out-of-fold predictions for model-audit purposes.

In [ ]:
legacy_builder = ARTIFACTS / 'build_features.py'
legacy_preparer = WORKING / 'prepare_legacy_stump_features.py'
for script in [legacy_builder, legacy_preparer, WORKING / 'run_best_blend.py',
               WORKING / 'train_stumps.py', WORKING / 'train_enriched_gam.py']:
    print('Running', script.name)
    script_env = os.environ.copy()
    script_env['LEGACY_STUMP_FEATURES'] = str(RUN_DIR / 'legacy_features' / 'final_dataset.csv.gz')
    subprocess.check_call([sys.executable, str(script)], cwd=ROOT, env=script_env)

for prediction_file in [
    WORKING / 'submission.csv',
    ARTIFACTS / 'stump_submission.csv',
    ARTIFACTS / 'enriched_gam_submission.csv',
]:
    assert prediction_file.exists(), f'Missing component prediction: {prediction_file}'
print('All component predictions created successfully.')

## 4. Build the final v19-style rank blend

Each component is converted to normalized rank order before blending. This avoids allowing different probability calibrations to dominate the ensemble while preserving each model’s ranking signal. The blend weights are 60% base blend, 10% shallow stump, and 30% enriched GAM-style model.

In [ ]:
import pandas as pd
from scipy.stats import rankdata

base = pd.read_csv(WORKING / 'submission.csv')
stump = pd.read_csv(ARTIFACTS / 'stump_submission.csv')
gam = pd.read_csv(ARTIFACTS / 'enriched_gam_submission.csv')

for candidate, label in [(stump, 'stump'), (gam, 'enriched GAM')]:
    assert candidate.encounter_id.tolist() == base.encounter_id.tolist(), (
        f'{label} predictions are not aligned with test encounter IDs.'
    )

def normalized_rank(values):
    return rankdata(values, method='average') / len(values)

final_prediction = (
    0.60 * normalized_rank(base.readmit_30d)
    + 0.10 * normalized_rank(stump.readmit_30d)
    + 0.30 * normalized_rank(gam.readmit_30d)
)
final_submission = pd.DataFrame({
    'encounter_id': base.encounter_id,
    'readmit_30d': final_prediction,
})
output_path = RUN_DIR / 'submission_v19_rank_old60_stump10_gam30.csv'
final_submission.to_csv(output_path, index=False)
print('Wrote:', output_path)

## 5. Validate the Kaggle submission

The final file must include every test `encounter_id` in the sample-submission order and a non-missing probability between 0 and 1 for `readmit_30d`.

In [ ]:
sample = pd.read_csv(WORKING / 'sample_submission.csv')
assert final_submission.columns.tolist() == ['encounter_id', 'readmit_30d']
assert final_submission.encounter_id.tolist() == sample.encounter_id.tolist()
assert len(final_submission) == len(sample) == 32812
assert final_submission.readmit_30d.notna().all()
assert final_submission.readmit_30d.between(0, 1).all()

print('Submission validation passed.')
print(final_submission.readmit_30d.describe().round(6))
final_submission.head()

## 6. Handoff notes

- The canonical submitted CSV is `final_submissions_and_artifacts/submission_v19_rank_old60_stump10_gam30.csv`.
- This notebook writes a separately reproduced file in `reproducibility_runs/` to protect the original submission artifact.
- `final_columns.csv` documents every feature used in the final modeling table and should be attached to the report appendix.
- Runtime is several hours on a machine with approximately 7 GB RAM because feature aggregation and five-fold model training are repeated.